<a href="https://colab.research.google.com/github/quyetttcoder/Fine-tune-LLM-with-small-data/blob/main/Quanntized_fine_tuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U gptqmodel optimum transformers[sentencepiece]
!pip install autoawq accelerate
!pip install huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 9.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... do

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_write_token = user_secrets.get_secret("HF_WRITE_TOKEN")

In [ ]:
from huggingface_hub import login
login(hf_write_token)

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_name = "quyetdev/llama31_8B_fine_tuned_16bit_v2"
quant_path = "llama31_8b_v2_awq"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load FP16 model
model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    device_map="cpu"
)

# Cấu hình quantization
quant_config = {
    "w_bit": 4,
    "q_group_size": 128,
    "zero_point": True,
    "version": "GEMM"
}

# Quantize
model.quantize(
    tokenizer,
    quant_config=quant_config
)

config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.
AWQ: 100%|██████████| 32/32 [45:19<00:00, 84.97s/it]


In [ ]:
# Save AWQ model
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

Writing model shards: 0it [00:00, ?it/s]

('llama31_8b_v2_awq/tokenizer_config.json',
 'llama31_8b_v2_awq/chat_template.jinja',
 'llama31_8b_v2_awq/tokenizer.json')

In [ ]:
from huggingface_hub import HfApi, upload_folder

repo_id = "quyetdev/llama31_8B_v2_fine_tuned_awq"

api = HfApi()

api.create_repo(repo_id, exist_ok=True)

upload_folder(
    repo_id=repo_id,
    folder_path=quant_path,
    commit_message="upload AWQ quantized model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/quyetdev/llama31_8B_v2_fine_tuned_awq/commit/57b0021852d7f7580584bbdbe61e7e2d7080447e', commit_message='upload AWQ quantized model', commit_description='', oid='57b0021852d7f7580584bbdbe61e7e2d7080447e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/quyetdev/llama31_8B_v2_fine_tuned_awq', endpoint='https://huggingface.co', repo_type='model', repo_id='quyetdev/llama31_8B_v2_fine_tuned_awq'), pr_revision=None, pr_num=None)